In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('Dataset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Dataset\gender_submission.csv
Dataset\test.csv
Dataset\train.csv


In [7]:
train = pd.read_csv(r'Dataset\train.csv')
test = pd.read_csv(r'Dataset\test.csv')
train.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


## Clean the Data. 

- see how many null values does the data have: <br>
```

print('Missing values Percentage: \n\n', round(df.isnull().sum().sort_values(ascending=False)/len(df)*100,1))

In [8]:
print('Missing values Percentage: \n\n', round (train.isnull().sum().sort_values(ascending=False)/len(train)*100,1))
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))


Missing values Percentage: 

 Cabin          77.1
Age            19.9
Embarked        0.2
PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
dtype: float64
Missing values Percentage: 

 Cabin          78.2
Age            20.6
Fare            0.2
PassengerId     0.0
Pclass          0.0
Name            0.0
Sex             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Embarked        0.0
dtype: float64


we see that the titles can be category of guesing the age of a person. like the title master. which can only be used for boys. and now we are going to extract the title and use it as one of hte methods filling up the age with mean values base on class, sex, parent (if they have or none) and title 

In [9]:
train['Titles'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Titles'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

In [10]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Titles
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,Mr
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,Mrs
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,Mr
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S,Mr
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S,Mrs


In [11]:
display(train.groupby(['Sex', 'Pclass', 'Parch', 'Titles'])['Age'].mean())

Sex     Pclass  Parch  Titles  
female  1       0      Countess    33.000000
                       Dr          49.000000
                       Lady        48.000000
                       Miss        34.965517
                       Mlle        24.000000
                       Mme         24.000000
                       Mrs         38.857143
                1      Miss        21.600000
                       Mrs         46.636364
                2      Miss        20.727273
                       Mrs         30.500000
        2       0      Miss        30.500000
                       Mrs         33.521739
                       Ms          28.000000
                1      Miss        10.285714
                       Mrs         33.818182
                2      Miss        10.833333
                       Mrs         32.000000
                3      Mrs         39.000000
        3       0      Miss        21.697674
                       Mrs         33.090909
                1      

we need to group titles like captain, col, major as one in order to remove sparsity. where a model learns less to feature that appears rarely. 

In [12]:
TitleDict = {"Capt": "Officer","Col": "Officer","Major": "Officer","Jonkheer": "Royalty", \
             "Don": "Royalty", "Sir" : "Royalty","Dr": "Royalty","Rev": "Royalty", \
             "Countess":"Royalty", "Mme": "Mrs", "Mlle": "Miss", "Ms": "Mrs","Mr" : "Mr", \
             "Mrs" : "Mrs","Miss" : "Miss","Master" : "Master","Lady" : "Royalty"}

train['Titles'] = train.Titles.map(TitleDict)
test['Titles'] = test.Titles.map(TitleDict)

In [13]:
display(train.groupby(['Sex', 'Pclass', 'Parch', 'Titles'])['Age'].mean())

Sex     Pclass  Parch  Titles 
female  1       0      Miss       34.258065
                       Mrs        38.181818
                       Royalty    43.333333
                1      Miss       21.600000
                       Mrs        46.636364
                2      Miss       20.727273
                       Mrs        30.500000
        2       0      Miss       30.500000
                       Mrs        33.291667
                1      Miss       10.285714
                       Mrs        33.818182
                2      Miss       10.833333
                       Mrs        32.000000
                3      Mrs        39.000000
        3       0      Miss       21.697674
                       Mrs        33.090909
                1      Miss        4.791667
                       Mrs        30.555556
                2      Miss        8.714286
                       Mrs        30.250000
                3      Mrs        36.000000
                4      Mrs        37.000000
 

In [14]:
# GROUP AND FIND THE AVERAGE 
train_master_means = train.groupby(['Sex', 'Pclass', 'Parch', 'Titles'])['Age'].transform('mean').round(2)
train_general_means = train.groupby(['Sex', 'Pclass', 'Parch'])['Age'].transform('mean').round(2)
train_global_means = train['Age'].mean().round(2)
# GROUP AND FIND THE AVERAGE 
test_master_means = test.groupby(['Sex', 'Pclass', 'Parch', 'Titles'])['Age'].transform('mean').round(2)
test_general_means = test.groupby(['Sex', 'Pclass', 'Parch'])['Age'].transform('mean').round(2)
test_global_means = test['Age'].mean().round(2)

In [15]:
# fill the none values in the age column 

test['Age'] = test['Age'].fillna(test_master_means)
test['Age'] = test['Age'].fillna(test_general_means)
test['Age'] = test['Age'].fillna(test_global_means)

train['Age'] = train['Age'].fillna(train_master_means)
train['Age'] = train['Age'].fillna(train_general_means)
train['Age'] = train['Age'].fillna(train_global_means)

In [16]:
train.sample(4)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Titles
621,622,1,1,"Kimball, Mr. Edwin Nelson Jr",male,42.0,1,0,11753,52.5542,D19,S,Mr
123,124,1,2,"Webber, Miss. Susan",female,32.5,0,0,27267,13.0000,E101,S,Miss
658,659,0,2,"Eitemiller, Mr. George Floyd",male,23.0,0,0,29751,13.0000,NaN,S,Mr
641,642,1,1,"Sagesser, Mlle. Emma",female,24.0,0,0,PC 17477,69.3000,B35,C,Miss


In [17]:
print('Missing values Percentage: \n\n', round (train.isnull().sum().sort_values(ascending=False)/len(train)*100,1))
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))


Missing values Percentage: 

 Cabin          77.1
Embarked        0.2
PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Titles          0.0
dtype: float64
Missing values Percentage: 

 Cabin          78.2
Fare            0.2
Titles          0.2
PassengerId     0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Embarked        0.0
dtype: float64


In [18]:
train['Embarked'] = train['Embarked'].fillna('U')

fillFare = test['Fare'].mean().round(2)
test['Fare'] = test['Fare'].fillna(fillFare)
train = train.drop(['Ticket', 'Cabin', 'PassengerId', 'Titles', 'Name'], axis=1)
test = test.drop(['Ticket', 'Cabin', 'PassengerId', 'Titles', 'Name'], axis=1)
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [20]:
print('Missing values Percentage: \n\n', round (train.isnull().sum().sort_values(ascending=False)/len(train)*100,1))
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))


Missing values Percentage: 

 Survived    0.0
Pclass      0.0
Sex         0.0
Age         0.0
SibSp       0.0
Parch       0.0
Fare        0.0
Embarked    0.0
dtype: float64
Missing values Percentage: 

 Pclass      0.0
Sex         0.0
Age         0.0
SibSp       0.0
Parch       0.0
Fare        0.0
Embarked    0.0
dtype: float64


In [21]:
train.shape, test.shape

((891, 8), (418, 7))